## Stage 2 - Temporal Transformer

### Data folder structure 

In [13]:
import os

for root, dirs, files in os.walk("JIGSAWS"):
    level = root.replace("JIGSAWS", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"    {indent}{f}")

JIGSAWS/
    .DS_Store
  Experimental_setup/
      .DS_Store
    Knot_Tying/
        .DS_Store
      Balanced/
        GestureClassification/
          OneTrialOut/
            25_Out/
              itr_44/
              itr_43/
              itr_17/
              itr_28/
              itr_10/
              itr_9/
              itr_26/
              itr_19/
              itr_21/
              itr_7/
              itr_42/
              itr_45/
              itr_6/
              itr_20/
              itr_27/
              itr_1/
              itr_18/
              itr_11/
              itr_8/
              itr_16/
              itr_29/
              itr_34/
              itr_33/
              itr_32/
              itr_35/
              itr_50/
              itr_13/
              itr_14/
              itr_22/
              itr_4/
              itr_3/
              itr_25/
              itr_49/
              itr_40/
              itr_47/
              itr_24/
              itr_2/
         

In [14]:
print(os.listdir("Cholec80"))
print(os.listdir("JIGSAWS"))

['.DS_Store', 'cholec80']
['.DS_Store', 'Experimental_setup', 'Knot_Tying', 'Suturing', 'Needle_Passing']


In [15]:
with open("JIGSAWS/Suturing/transcriptions/Suturing_B001.txt") as f:
    print(f.read())

80 219 G1 
220 370 G5 
371 590 G8 
591 660 G2 
661 954 G3 
955 1097 G8 
1098 1124 G2 
1125 1401 G3 
1402 1439 G2 
1440 1698 G8 
1699 1783 G2 
1784 2285 G3 
2286 2495 G6 
2496 2679 G4 
2680 2750 G2 
2751 2976 G3 
2977 3155 G6 
3156 3308 G4 
3309 3659 G2 
3660 4011 G3 
4012 4321 G8 
4322 4374 G2 
4375 4704 G3 
4705 4846 G6 
4847 4983 G4 
4984 5059 G2 
5060 5354 G3 
5355 5447 G6 
5448 5635 G11 



In [16]:
videos = os.listdir("JIGSAWS/Suturing/video/")
print(sorted(videos)[:5])
print(f"Total videos: {len(videos)}")

['Suturing_B001_capture1.avi', 'Suturing_B001_capture2.avi', 'Suturing_B002_capture1.avi', 'Suturing_B002_capture2.avi', 'Suturing_B003_capture1.avi']
Total videos: 78


### Dataloader

In [17]:
import torch 
from torch.utils.data import Dataset, DataLoader
import av
import numpy as np 
import os
from torchvision import transforms
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ensures deterministic behavior in multi-head attention
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# mapping gestures to numbers
GESTURE_MAP = {
    'G1':0,'G2':1,'G3':2,'G4':3,'G5':4,
    'G6':5,'G8':6,'G9':7,'G10':8,'G11':9
}

In [18]:
class JIGSAWSClipDataset(Dataset):
    def __init__(self, task='Suturing', split_trials=None, clip_len=8):
        # split_trials is the list of trial names to include like 'B001', 'B002'

        self.clip_len = clip_len
        self.clips = [] # will (store video_path, start, end, label)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        # looping through every transcription file - one per video
        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue

            # extract trial name
            trial = fname.replace(f'{task}_', '').replace('.txt', '')

            # test trials skipped
            if split_trials and trial not in split_trials:
                continue
            
            # we use capture 1 camera angle
            video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi") 
            if not os.path.exists(video_path):
                continue

            # read transcription file
            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3: continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]

                    # keeping only gestures we know and long enough to sample 8 frames
                    if gesture in GESTURE_MAP and (end-start)>clip_len:
                        self.clips.append((video_path, start, end, GESTURE_MAP[gesture]))

                
        # transform to match ResNet
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias = True),
            transforms.Normalize([0.485,0.456,0.406],
                                [0.229,0.224,0.225]) 
        ])
        print(f"    {task} {split_trials}: {len(self.clips)} gesture clips")

    
    def _load_frames(self, video_path, start, end):
        # pick clip_len frame indices evenly spread across the gesture segment

        frame_ids = set(np.linspace(start, end, self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path) # open the video file
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                img = frame.to_ndarray(format='rgb24') # converting to RGB numpy array
                frames.append(self.transform(img)) # resize + transform
            if len(frames) == self.clip_len:
                break # stop when we have all frames

        container.close()

        while len(frames) < self.clip_len:
            frames.append(frames[-1]) # pad the last frame if less

        return torch.stack(frames) # stack into (T, C, H, W) tensor

    def __len__(self): return len(self.clips)

    def __getitem__(self,idx):
        video_path, start, end, label = self.clips[idx]
        frames = self._load_frames(video_path, start, end)

        # for stage 2 we return all frames
        return frames, label            

In [19]:
# loading a sample to check

ds = JIGSAWSClipDataset(task="Suturing")
img, label = ds[0]
print(f"Sample shape: {img.shape}, label: {label}, gesture: {list(GESTURE_MAP.keys())[label]}")

    Suturing None: 793 gesture clips
Sample shape: torch.Size([8, 3, 224, 224]), label: 0, gesture: G1


### Train/test split

In [20]:
# get all trial names from the transciptions folder
all_trials = [f.replace('Suturing_', '').replace('.txt','')
            for f in os.listdir('JIGSAWS/Suturing/transcriptions/')
            if f.endswith('.txt')]
all_trials = sorted(all_trials)
print("All trials:", all_trials)

# first four trials for test rest train
# JIGSAWS uses leave-one-user-out i am doing it in later stages

test_trials = all_trials[:4]
train_trials = all_trials[4:]

print(f"Train: {train_trials}")
print(f"Test: {test_trials}")

#  create dataset objects
train_clip_ds = JIGSAWSClipDataset('Suturing', split_trials=train_trials)
test_clip_ds  = JIGSAWSClipDataset('Suturing', split_trials=test_trials)

# dataloadaer handles batching + shuffling
train_clip_loader = DataLoader(train_clip_ds, batch_size=16, shuffle=True, num_workers=0)
test_clip_loader = DataLoader(test_clip_ds, batch_size=16, shuffle=False, num_workers=0)
print(f"\nTrain batches: {len(train_clip_loader)}, Test batches: {len(test_clip_loader)}")

All trials: ['B001', 'B002', 'B003', 'B004', 'B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Train: ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Test: ['B001', 'B002', 'B003', 'B004']
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Suturing ['B001', 'B002', 'B003

### Temporal Transformer model

In [21]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class SurgicalTransformer(nn.Module):
    def __init__(self, n_classes=10, clip_len=8, feat_dim=512, 
                n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len

        # resnet50 but strip the final layer to output 2048-dim features instead of class scores
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.frame_encoder = nn.Sequential(*list(resnet.children())[:-1])

        # project 2048 -> 512 smaller and faster transformer input
        self.proj = nn.Linear(2048, feat_dim)

        # temporal transformer
        # this accumulates the summary of the entire clip
        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))

        # give it the postional embedding as well
        self.pos_embed = nn.Parameter(torch.randn(1, clip_len+1, feat_dim))

        # 4 layer transformer encoder... each frame attends to every other frame
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm = nn.LayerNorm(feat_dim)

        # classifier
        self.head = nn.Linear(feat_dim, n_classes)

    
    def forward(self, x):
        # x is (B, T, C, H, W) - batch of clips
        # returns (B, n_classes) - logits per clip

        B, T, C, H, W = x.shape

        # extract features from every fram independently
        # reshape so all frames are processed at once

        x = x.view(B*T, C, H, W)
        x = self.frame_encoder(x)
        x = x.view(B*T, -1)
        x = self.proj(x)
        x = x.view(B, T, -1)

        # prepend the cls token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # add pos embeddings
        x = x + self.pos_embed

        # transformer
        x = self.transformer(x)
        x = self.norm(x)

        # cls token at pos 0 
        clip_repr = x[:, 0, :]

        return self.head(clip_repr)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model2 = SurgicalTransformer(n_classes=10, clip_len=8).to(device)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=5, gamma=0.5)

total = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}")
print(f"Device: {device}")


Trainable params: 37,177,930
Device: mps


### Training Loop

In [ ]:
from sklearn.metrics import f1_score
import time

def run_epoch2(loader, train=True):
    model2.train() if train else model2.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.set_grad_enabled(train):
        for clips, labels in loader:

            # full clips now not single frames (B, T, C, H, W)
            clips, labels = clips.to(device), labels.to(device)

            out = model2(clips)
            loss = criterion2(out, labels)

            if train:
                optimizer2.zero_grad()
                loss.backward()
                # gradient clipping prevents exploding grads in transformers
                torch.nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
                optimizer2.step()

            total_loss += loss.item()
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        return total_loss / len(loader), f1

print("Stage 2 training - Resnet50 + Temporal Transformer")
print("="*50)

best_test_f1 = 0
for epoch in range(1,16):
    t0 = time.time()
    tr_loss, tr_f1 = run_epoch2(train_clip_loader, train=True)
    te_loss, te_f1 = run_epoch2(test_clip_loader, train=False)
    scheduler2.step()

    if te_f1 > best_test_f1:
        best_test_f1 = te_f1
        # torch.save(model2.state_dict(), 'stage2_best.pt')

    print(f"Epoch {epoch:2d} | "
          f"Train loss: {tr_loss:.3f} F1: {tr_f1:.3f} | "
          f"Test  loss: {te_loss:.3f} F1: {te_f1:.3f} | "
          f"{time.time()-t0:.1f}s")

print(f"Stage 2 Best Test F1: {best_test_f1:.3f}")

Stage 2 training - Resnet50 + Temporal Transformer
Epoch  1 | Train loss: 1.905 F1: 0.152 | Test  loss: 1.626 F1: 0.356 | 182.8s
Epoch  2 | Train loss: 0.469 F1: 0.691 | Test  loss: 0.767 F1: 0.925 | 188.7s
Epoch  3 | Train loss: 0.184 F1: 0.839 | Test  loss: 0.969 F1: 0.726 | 181.9s
Epoch  4 | Train loss: 0.171 F1: 0.835 | Test  loss: 0.779 F1: 0.817 | 180.3s
Epoch  5 | Train loss: 0.106 F1: 0.866 | Test  loss: 0.827 F1: 0.782 | 180.0s
Epoch  6 | Train loss: 0.050 F1: 0.967 | Test  loss: 0.779 F1: 0.857 | 178.1s
Epoch  7 | Train loss: 0.014 F1: 0.963 | Test  loss: 0.533 F1: 0.918 | 179.0s
Epoch  8 | Train loss: 0.014 F1: 0.983 | Test  loss: 0.536 F1: 0.878 | 180.7s
Epoch  9 | Train loss: 0.008 F1: 0.974 | Test  loss: 0.613 F1: 0.887 | 189.6s
Epoch 10 | Train loss: 0.002 F1: 1.000 | Test  loss: 0.668 F1: 0.924 | 187.1s
Epoch 11 | Train loss: 0.001 F1: 1.000 | Test  loss: 0.686 F1: 0.903 | 187.4s
Epoch 12 | Train loss: 0.001 F1: 1.000 | Test  loss: 0.674 F1: 0.924 | 188.8s
Epoch 13 | Tr

In [11]:
tasks = ['Knot_Tying', 'Needle_Passing']

task2_results = {}

for task in tasks:
    print(f"\n---   {task}  ---")

    task_train_ds = JIGSAWSClipDataset(task, split_trials=train_trials)
    task_test_ds  = JIGSAWSClipDataset(task, split_trials=test_trials)

    if len(task_test_ds) == 0:
        print(" Skipping - no test clips")
        continue

    task_train_loader = DataLoader(task_train_ds, batch_size=16, shuffle=True,  num_workers=0)
    task_test_loader  = DataLoader(task_test_ds,  batch_size=16, shuffle=False, num_workers=0)

    tm = SurgicalTransformer(n_classes=10, clip_len=8).to(device)
    tc = nn.CrossEntropyLoss()
    to = torch.optim.Adam(tm.parameters(), lr=1e-4, weight_decay=1e-4)
    ts = torch.optim.lr_scheduler.StepLR(to, step_size=5, gamma=0.5)

    best_f1 = 0
    for epoch in range(1, 16):
        # train
        tm.train()
        tr_preds, tr_labels = [], []
        for clips, labels in task_train_loader:
            clips, labels = clips.to(device), labels.to(device)
            out  = tm(clips)
            loss = tc(out, labels)
            to.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tm.parameters(), max_norm=1.0)
            to.step()
            tr_preds.extend(out.argmax(1).cpu().numpy())
            tr_labels.extend(labels.cpu().numpy())
        ts.step()
        tr_f1 = f1_score(tr_labels, tr_preds, average='macro', zero_division=0)

        # evaluate
        tm.eval()
        te_preds, te_labels = [], []
        with torch.no_grad():
            for clips, labels in task_test_loader:
                clips, labels = clips.to(device), labels.to(device)
                out = tm(clips)
                te_preds.extend(out.argmax(1).cpu().numpy())
                te_labels.extend(labels.cpu().numpy())
        te_f1 = f1_score(te_labels, te_preds, average='macro', zero_division=0)

        if te_f1 > best_f1:
            best_f1 = te_f1

        print(f"  Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f}")

    task2_results[task] = best_f1

# summary
print("\n" + "="*50)
print("STAGE 2 SUMMARY — Temporal Transformer")
print("="*50)
for task, f1 in task2_results.items():
    print(f"  {task:<20}: Best Test F1 = {f1:.3f}")


---   Knot_Tying  ---
    Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 gesture clips
    Knot_Tying ['B001', 'B002', 'B003', 'B004']: 4 gesture clips
  Epoch  1 | Train F1: 0.163 | Test F1: 0.000
  Epoch  2 | Train F1: 0.879 | Test F1: 1.000
  Epoch  3 | Train F1: 1.000 | Test F1: 1.000
  Epoch  4 | Train F1: 1.000 | Test F1: 1.000
  Epoch  5 | Train F1: 1.000 | Test F1: 1.000
  Epoch  6 | Train F1: 1.000 | Test F1: 1.000
  Epoch  7 | Train F1: 1.000 | Test F1: 1.000
  Epoch  8 | Train F1: 1.000 | Test F1: 1.000
  Epoch  9 | Train F1: 1.000 | Test F1: 1.000
  Epoch 10 | Train F1: 1.000 | Test F1: 1.000
  Epoch 11 | Train F1: 1.000 | Test F1: 1.000
  Epoch 12 | Train F1: 1.000 | Test F1: 1.000
  Epoch 13 | Train F1: 1.000 | Test F1: 1.000


In [22]:
from torch.utils.data import ConcatDataset

tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']

train_clip_ds = ConcatDataset([JIGSAWSClipDataset(t, split_trials=train_trials) for t in tasks])
test_clip_ds  = ConcatDataset([JIGSAWSClipDataset(t, split_trials=test_trials)  for t in tasks])

train_clip_loader = DataLoader(train_clip_ds, batch_size=8, shuffle=True,  num_workers=0)
test_clip_loader  = DataLoader(test_clip_ds,  batch_size=8, shuffle=False, num_workers=0)

print(f"Total train clips: {len(train_clip_ds)}")
print(f"Total test clips:  {len(test_clip_ds)}")
print(f"Train batches: {len(train_clip_loader)}, Test batches: {len(test_clip_loader)}")

    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 gesture clips
    Needle_Passing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 453 gesture clips
    Suturing ['B001', 'B002', 'B003', 'B004']: 83 gest

In [23]:
from sklearn.metrics import f1_score
import time

print("Stage 2 - Combined (all 3 tasks)")
print("="*50)

best_test_f1 = 0

model2 = SurgicalTransformer(n_classes=10, clip_len=8).to(device)
criterion2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=5, gamma=0.5)

for epoch in range(1, 16):
    t0 = time.time()

    # train
    model2.train()
    tr_preds, tr_labels = [], []
    for clips, labels in train_clip_loader:
        clips, labels = clips.to(device), labels.to(device)
        out  = model2(clips)
        loss = criterion2(out, labels)
        optimizer2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
        optimizer2.step()
        tr_preds.extend(out.argmax(1).cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())
    scheduler2.step()
    tr_f1 = f1_score(tr_labels, tr_preds, average='macro', zero_division=0)

    # evaluate
    model2.eval()
    te_preds, te_labels = [], []
    with torch.no_grad():
        for clips, labels in test_clip_loader:
            clips, labels = clips.to(device), labels.to(device)
            out = model2(clips)
            te_preds.extend(out.argmax(1).cpu().numpy())
            te_labels.extend(labels.cpu().numpy())
    te_f1 = f1_score(te_labels, te_preds, average='macro', zero_division=0)

    if te_f1 > best_test_f1:
        best_test_f1 = te_f1
        torch.save(model2.state_dict(), 'stage2_combined_best.pt')

    print(f"Epoch {epoch:2d} | "
          f"Train F1: {tr_f1:.3f} | "
          f"Test F1: {te_f1:.3f} | "
          f"{time.time()-t0:.1f}s")

print(f"\nBest Test F1 (combined): {best_test_f1:.3f}")

Stage 2 - Combined (all 3 tasks)
Epoch  1 | Train F1: 0.216 | Test F1: 0.572 | 291.5s
Epoch  2 | Train F1: 0.639 | Test F1: 0.642 | 295.2s
Epoch  3 | Train F1: 0.751 | Test F1: 0.700 | 295.4s
Epoch  4 | Train F1: 0.809 | Test F1: 0.728 | 294.8s
Epoch  5 | Train F1: 0.809 | Test F1: 0.690 | 295.4s
Epoch  6 | Train F1: 0.854 | Test F1: 0.646 | 295.5s
Epoch  7 | Train F1: 0.932 | Test F1: 0.746 | 295.8s
Epoch  8 | Train F1: 0.941 | Test F1: 0.733 | 295.7s
Epoch  9 | Train F1: 0.973 | Test F1: 0.684 | 296.1s
Epoch 10 | Train F1: 0.966 | Test F1: 0.611 | 296.2s
Epoch 11 | Train F1: 0.955 | Test F1: 0.606 | 296.4s
Epoch 12 | Train F1: 1.000 | Test F1: 0.594 | 296.4s
Epoch 13 | Train F1: 1.000 | Test F1: 0.606 | 297.0s
Epoch 14 | Train F1: 1.000 | Test F1: 0.607 | 296.8s
Epoch 15 | Train F1: 1.000 | Test F1: 0.592 | 297.4s

Best Test F1 (combined): 0.746
